# Tutorial de spaCy
### Procesamiento de Lenguaje Natural en Python

**Materia:** Métodos de Texto (NLP) &nbsp;|&nbsp; **Tarea No. 2: Librerías básicas de NLP en Python**

**Equipo 3 · spaCy**
*Lambda Remiel Heredia Pérez y Diego Octavio Pérez Urbina*

---

Este notebook es un tutorial autocontenido para **aprender a usar spaCy desde cero** y tener
una base sólida para empezar a desarrollar programas de procesamiento de lenguaje natural (NLP).
Incluye explicación conceptual + ejemplos de código ejecutados y comentados para cada función básica
de la librería. También es el material que usaremos para la presentación de ~30 minutos: se
recorre en vivo, de principio a fin, directamente en Jupyter.

**Contenido:**

1. ¿Qué es spaCy?
2. Instalación y modelos de idioma
3. Cargando un modelo: el objeto `nlp` y el *pipeline*
4. El objeto `Doc` y la tokenización
5. Atributos lingüísticos de cada `Token`
6. Palabras vacías (*stop words*)
7. Lematización
8. Segmentación de oraciones
9. Análisis de dependencias sintácticas
10. Reconocimiento de entidades nombradas (NER)
11. Sintagmas nominales (`noun_chunks`)
12. Vectores de palabras y similitud semántica
13. Búsqueda de patrones: `Matcher` y `PhraseMatcher`
14. Procesamiento eficiente de muchos textos: `nlp.pipe`
15. Personalizando el *pipeline*: componentes y extensiones propias
16. Serialización: guardar y cargar resultados (`DocBin`)
17. Proyecto integrador: extractor de información de una noticia
18. Buenas prácticas, limitaciones y siguientes pasos
19. Resumen / *cheatsheet* de referencia rápida

## 1. ¿Qué es spaCy?

[spaCy](https://spacy.io) es una librería de código abierto para **Procesamiento de Lenguaje
Natural (NLP)** en Python, diseñada para uso en **producción** (a diferencia de NLTK, que nació
como herramienta docente/de investigación). Sus características principales son:

- **Rápida y eficiente**: el núcleo está escrito en Cython.
- **Modelos estadísticos preentrenados** para decenas de idiomas (incluido español), que permiten
  hacer etiquetado gramatical, análisis sintáctico y reconocimiento de entidades "de fábrica".
- **Pipelines configurables**: el procesamiento de un texto se hace en pasos (tokenización → POS →
  parser → NER → ...) y cada paso se puede activar, desactivar o reemplazar.
- **Objetos ricos** (`Doc`, `Token`, `Span`) que representan el texto ya analizado, en lugar de
  simples listas de strings.
- Soporte para **vectores de palabras**, **coincidencia de patrones basada en reglas**, y
  entrenamiento de **modelos propios** (incluyendo *transformers*).

### ¿spaCy vs. NLTK vs. re/regex?

| | `re` / regex | NLTK | spaCy |
|---|---|---|---|
| Enfoque | Patrones de texto plano | Docencia / investigación, muy modular | Producción, rápido, "todo incluido" |
| Modelos preentrenados | No | Algunos | Sí (varios idiomas y tamaños) |
| Objeto de análisis | strings | strings / árboles | objetos `Doc`/`Token`/`Span` enlazados |
| Velocidad | Alta (pero sin lingüística) | Media | Alta |

En este tutorial usaremos principalmente **español** (`es_core_news_sm` / `es_core_news_md`),
pero todo lo que veremos aplica igual para inglés (`en_core_web_sm`) u otros idiomas.

## 2. Instalación y modelos de idioma

spaCy se instala con `pip`. Los modelos de idioma (los pesos ya entrenados) se descargan aparte,
porque no es práctico incluir en la librería modelos para decenas de idiomas.

Ejecuta esto **una sola vez** (en una terminal o descomentando las líneas en Jupyter con `!`):

In [1]:
# Instalación de la librería (ejecutar una sola vez)
# !pip install -U spacy

# Descarga de modelos de idioma (elige los que necesites)
# !python -m spacy download es_core_news_sm   # español, pequeño y rápido
# !python -m spacy download es_core_news_md   # español, incluye vectores de palabras
# !python -m spacy download en_core_web_sm    # inglés, pequeño

Los modelos siguen la convención `{idioma}_{género}_{tamaño}`:

- `es` / `en` → idioma (español / inglés).
- `core_news` → modelo genérico entrenado con noticias; `core_web` → entrenado con texto de la web.
- `sm` (small), `md` (medium, con vectores de palabras), `lg` (large), `trf` (basado en *transformers*,
  el más preciso pero más pesado).

Para este tutorial usaremos `es_core_news_sm` (rápido, ideal para desarrollo) y `es_core_news_md`
únicamente en la sección de similitud semántica (necesita vectores de palabras reales).

In [2]:
import spacy

print("Versión de spaCy:", spacy.__version__)

Versión de spaCy: 3.8.16


## 3. Cargando un modelo: el objeto `nlp` y el *pipeline*

`spacy.load(...)` carga un modelo y regresa un objeto `nlp`: una función/objeto que convierte
texto plano en un objeto `Doc` totalmente analizado.

Al llamar `nlp(texto)`, spaCy pasa el texto por una serie de pasos encadenados, el **pipeline**:
tokenizador → *tagger*/*morphologizer* (gramática) → *parser* (sintaxis) → *lemmatizer* → *NER*
(entidades) → ... El tokenizador siempre corre primero y de forma especial; el resto de
componentes se pueden inspeccionar, quitar o sustituir (lo veremos en la sección 15).

In [3]:
nlp = spacy.load("es_core_news_sm")

texto = "Apple está buscando comprar una startup del Reino Unido por mil millones de dólares."
doc = nlp(texto)

print(type(doc))
print(doc)

<class 'spacy.tokens.doc.Doc'>
Apple está buscando comprar una startup del Reino Unido por mil millones de dólares.


In [4]:
# ¿Qué componentes tiene el pipeline y en qué orden corren?
print("Componentes:", nlp.pipe_names)
for nombre, componente in nlp.pipeline:
    print(f" - {nombre}: {componente}")

Componentes: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
 - tok2vec: <spacy.pipeline.tok2vec.Tok2Vec object at 0x0000015F1C0A63F0>
 - morphologizer: <spacy.pipeline.morphologizer.Morphologizer object at 0x0000015F1C0A6630>
 - parser: <spacy.pipeline.dep_parser.DependencyParser object at 0x0000015F1C151380>
 - attribute_ruler: <spacy.pipeline.attributeruler.AttributeRuler object at 0x0000015F1C18DD90>
 - lemmatizer: <spacy.lang.es.lemmatizer.SpanishLemmatizer object at 0x0000015F1C18E910>
 - ner: <spacy.pipeline.ner.EntityRecognizer object at 0x0000015F1C151690>


## 4. El objeto `Doc` y la tokenización

**Tokenizar** es dividir un texto en unidades mínimas con sentido (*tokens*): palabras, números,
signos de puntuación, etc. spaCy tokeniza de forma **no destructiva**: siempre se puede reconstruir
el texto original a partir de sus tokens (incluye espacios, ver `token.whitespace_`).

Un `Doc` se comporta como una **secuencia de objetos `Token`**: se puede iterar, indexar y
"rebanar" (slicing) igual que una lista.

In [5]:
for token in doc:
    print(token.i, repr(token.text))

0 'Apple'
1 'está'
2 'buscando'
3 'comprar'
4 'una'
5 'startup'
6 'del'
7 'Reino'
8 'Unido'
9 'por'
10 'mil'
11 'millones'
12 'de'
13 'dólares'
14 '.'


In [6]:
# Indexar un Doc regresa un Token; "rebanarlo" regresa un Span (varios tokens)
print("Token 0:", doc[0], "->", type(doc[0]))
print("Span 3:6:", doc[3:6], "->", type(doc[3:6]))

# El texto original siempre se puede reconstruir
print("¿Se reconstruye igual?", doc.text == texto)

Token 0: Apple -> <class 'spacy.tokens.token.Token'>
Span 3:6: comprar una startup -> <class 'spacy.tokens.span.Span'>
¿Se reconstruye igual? True


## 5. Atributos lingüísticos de cada `Token`

Cada `Token` ya trae "de fábrica" mucha información lingüística calculada por el pipeline:

| Atributo | Significado |
|---|---|
| `token.text` | El texto literal del token |
| `token.lemma_` | El lema (forma base / diccionario) |
| `token.pos_` | Categoría gramatical universal (VERB, NOUN, ADJ, ...) |
| `token.tag_` | Etiqueta gramatical detallada (depende del idioma) |
| `token.dep_` | Función sintáctica dentro de la oración (relación de dependencia) |
| `token.is_stop` | ¿Es una palabra vacía / de relleno? |
| `token.is_alpha` | ¿Son solo letras? |
| `token.is_punct` | ¿Es un signo de puntuación? |
| `token.like_num` | ¿Parece un número? |
| `token.shape_` | Forma ortográfica (p.ej. `Xxxxx`, `dd`) |

`spacy.explain("XXX")` sirve para traducir cualquier etiqueta abreviada a una descripción legible.

In [7]:
print(f"{'TEXTO':12}{'LEMA':12}{'POS':8}{'DEP':8}{'STOP':6}")
for token in doc:
    print(f"{token.text:12}{token.lemma_:12}{token.pos_:8}{token.dep_:8}{str(token.is_stop):6}")

TEXTO       LEMA        POS     DEP     STOP  
Apple       Apple       PROPN   nsubj   False 
está        estar       AUX     aux     True  
buscando    buscar      VERB    ROOT    False 
comprar     comprar     VERB    xcomp   False 
una         uno         DET     det     True  
startup     startup     NOUN    obj     False 
del         del         ADP     case    True  
Reino       Reino       PROPN   nmod    False 
Unido       Unido       PROPN   flat    False 
por         por         ADP     case    True  
mil         mil         NUM     nummod  False 
millones    millón      NOUN    obl     False 
de          de          ADP     case    True  
dólares     dólares     NOUN    nmod    False 
.           .           PUNCT   punct   False 


In [8]:
# spacy.explain traduce las etiquetas abreviadas
for etiqueta in ["VERB", "PROPN", "obj", "nsubj", "AUX"]:
    print(f"{etiqueta:8} -> {spacy.explain(etiqueta)}")

VERB     -> verb
PROPN    -> proper noun
obj      -> object
nsubj    -> nominal subject
AUX      -> auxiliary


## 6. Palabras vacías (*stop words*)

Las *stop words* son palabras muy frecuentes y con poco contenido semántico propio
(artículos, preposiciones, pronombres...). spaCy trae una lista predefinida por idioma, y cada
token ya sabe si es *stop word* gracias a `token.is_stop`.

In [9]:
from spacy.lang.es.stop_words import STOP_WORDS

print("Número de stop words en español:", len(STOP_WORDS))
print("Algunos ejemplos:", sorted(list(STOP_WORDS))[:10])

Número de stop words en español: 521
Algunos ejemplos: ['a', 'acuerdo', 'adelante', 'ademas', 'además', 'afirmó', 'agregó', 'ahi', 'ahora', 'ahí']


In [10]:
# Quitar stop words y puntuación de una oración (útil para bolsas de palabras, TF-IDF, etc.)
palabras_clave = [t.text for t in doc if not t.is_stop and not t.is_punct]
print("Original: ", [t.text for t in doc])
print("Sin stop words ni puntuación:", palabras_clave)

Original:  ['Apple', 'está', 'buscando', 'comprar', 'una', 'startup', 'del', 'Reino', 'Unido', 'por', 'mil', 'millones', 'de', 'dólares', '.']
Sin stop words ni puntuación: ['Apple', 'buscando', 'comprar', 'startup', 'Reino', 'Unido', 'mil', 'millones', 'dólares']


## 7. Lematización

**Lematizar** es reducir una palabra a su forma canónica de diccionario (el *lema*), tomando en
cuenta su categoría gramatical y contexto, a diferencia del *stemming* (que solo recorta
sufijos con reglas, sin garantizar que el resultado sea una palabra real). spaCy **no** incluye
*stemmer*; su enfoque es siempre la lematización basada en reglas + morfología, que es más precisa
para idiomas con mucha flexión como el español.

In [11]:
frases = ["corro todos los días", "corrí un maratón", "correremos mañana", "estamos corriendo"]
for frase in frases:
    d = nlp(frase)
    lemas = [(t.text, t.lemma_) for t in d if t.pos_ == "VERB"]
    print(f"{frase!r:32} -> {lemas}")

'corro todos los días'           -> [('corro', 'correr')]
'corrí un maratón'               -> [('corrí', 'corrí')]
'correremos mañana'              -> [('correremos', 'correr')]
'estamos corriendo'              -> [('corriendo', 'correr')]


## 8. Segmentación de oraciones

Un `Doc` también se puede recorrer por oraciones con `doc.sents` (cada oración es un `Span`).
La detección de límites de oración depende del *parser* sintáctico (o de un componente `senter`
más ligero), no de una simple búsqueda de puntos.

In [12]:
parrafo = ("Vivo en Guadalajara. Estudio en el ITESO. "
           "Me interesa el procesamiento de lenguaje natural, sobre todo con spaCy.")
doc_parrafo = nlp(parrafo)

for i, oracion in enumerate(doc_parrafo.sents, start=1):
    print(f"Oración {i}: {oracion.text!r}")

Oración 1: 'Vivo en Guadalajara.'
Oración 2: 'Estudio en el ITESO.'
Oración 3: 'Me interesa el procesamiento de lenguaje natural, sobre todo con spaCy.'


## 9. Análisis de dependencias sintácticas

El *parser* de dependencias conecta cada token con su "cabeza" (`token.head`) mediante una
etiqueta de relación (`token.dep_`): sujeto, objeto, modificador, etc. Esto forma un árbol
sintáctico que se puede recorrer con `token.children` (hijos) y `token.ancestors` (ancestros).

spaCy incluye un visualizador integrado, **displaCy**, que dibuja este árbol.

In [13]:
for token in doc:
    hijos = [hijo.text for hijo in token.children]
    print(f"{token.text:12} dep={token.dep_:8} head={token.head.text:12} hijos={hijos}")

Apple        dep=nsubj    head=buscando     hijos=[]
está         dep=aux      head=buscando     hijos=[]
buscando     dep=ROOT     head=buscando     hijos=['Apple', 'está', 'comprar', '.']
comprar      dep=xcomp    head=buscando     hijos=['startup', 'millones']
una          dep=det      head=startup      hijos=[]
startup      dep=obj      head=comprar      hijos=['una', 'Reino']
del          dep=case     head=Reino        hijos=[]
Reino        dep=nmod     head=startup      hijos=['del', 'Unido']
Unido        dep=flat     head=Reino        hijos=[]
por          dep=case     head=millones     hijos=[]
mil          dep=nummod   head=millones     hijos=[]
millones     dep=obl      head=comprar      hijos=['por', 'mil', 'dólares']
de           dep=case     head=dólares      hijos=[]
dólares      dep=nmod     head=millones     hijos=['de']
.            dep=punct    head=buscando     hijos=[]


In [14]:
from spacy import displacy

# En Jupyter, jupyter=True despliega el gráfico directamente debajo de la celda
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 100})

## 10. Reconocimiento de entidades nombradas (NER)

El componente `ner` identifica **entidades del mundo real** mencionadas en el texto (personas,
organizaciones, lugares, etc.) y las expone en `doc.ents` como objetos `Span`, cada uno con
`.text` y `.label_`.

> ⚠️ El conjunto de etiquetas depende del **modelo**, no de spaCy en general. Los modelos en
> español (`es_core_news_*`) usan un esquema más simple (`PER`, `LOC`, `ORG`, `MISC`) que los
> modelos en inglés (`en_core_web_*`), que además distinguen `DATE`, `MONEY`, `GPE`, `PRODUCT`, etc.

In [15]:
for ent in doc.ents:
    print(f"{ent.text:25} {ent.label_:6} {spacy.explain(ent.label_)}")

Apple                     ORG    Companies, agencies, institutions, etc.
Reino Unido               LOC    Non-GPE locations, mountain ranges, bodies of water


In [16]:
displacy.render(doc, style="ent", jupyter=True)

In [17]:
# Probemos con un texto con más variedad de entidades
doc_noticia = nlp("Tim Cook viajó a Madrid el 15 de enero para reunirse con directivos del BBVA "
                   "y de Iberdrola. Lionel Messi, por su parte, sigue jugando en el Inter Miami.")
for ent in doc_noticia.ents:
    print(f"{ent.text:20} {ent.label_}")

Tim Cook             PER
Madrid               LOC
BBVA                 ORG
Iberdrola            ORG
Lionel Messi         PER
Inter Miami          ORG


## 11. Sintagmas nominales (`noun_chunks`)

`doc.noun_chunks` regresa grupos de palabras que giran alrededor de un sustantivo (sujeto,
objetos, complementos...), útiles para resumir "de qué habla" una oración sin analizar
dependencia por dependencia.

In [18]:
for chunk in doc.noun_chunks:
    print(f"{chunk.text:20} | raíz: {chunk.root.text:10} dep={chunk.root.dep_:6} head={chunk.root.head.text}")

Apple                | raíz: Apple      dep=nsubj  head=buscando
una startup          | raíz: startup    dep=obj    head=comprar
Reino Unido          | raíz: Reino      dep=nmod   head=startup
mil millones         | raíz: millones   dep=obl    head=comprar
dólares              | raíz: dólares    dep=nmod   head=millones


## 12. Vectores de palabras y similitud semántica

Los modelos `md` y `lg` incluyen **vectores de palabras** (*word embeddings*): representaciones
numéricas donde palabras con significado parecido quedan "cerca" en el espacio vectorial. Esto
permite calcular una **similitud semántica** entre tokens, *spans* o documentos completos con
`.similarity()` (basada en similitud coseno).

> El modelo `sm` (small) **no** trae vectores de palabras reales (solo representaciones internas
> del tokenizador), así que para esta sección cargamos `es_core_news_md`.

In [19]:
nlp_md = spacy.load("es_core_news_md")

palabras = nlp_md("perro gato plátano")
for palabra in palabras:
    print(palabra.text, "-> vector de dimensión", palabra.vector.shape)

print()
for a in palabras:
    for b in palabras:
        print(f"similitud({a.text}, {b.text}) = {a.similarity(b):.3f}")

perro -> vector de dimensión (300,)
gato -> vector de dimensión (300,)
plátano -> vector de dimensión (300,)

similitud(perro, perro) = 1.000
similitud(perro, gato) = 0.849
similitud(perro, plátano) = 0.271
similitud(gato, perro) = 0.849
similitud(gato, gato) = 1.000
similitud(gato, plátano) = 0.281
similitud(plátano, perro) = 0.271
similitud(plátano, gato) = 0.281
similitud(plátano, plátano) = 1.000


In [20]:
# La similitud también funciona a nivel de documento completo (promedio de vectores)
doc_a = nlp_md("Me encanta programar en Python")
doc_b = nlp_md("Python es mi lenguaje de programación favorito")
doc_c = nlp_md("Hoy va a llover en la tarde")

print("a vs b (deberían parecerse):", round(doc_a.similarity(doc_b), 3))
print("a vs c (no deberían parecerse):", round(doc_a.similarity(doc_c), 3))

a vs b (deberían parecerse): 0.321
a vs c (no deberían parecerse): 0.11


## 13. Búsqueda de patrones basada en reglas: `Matcher` y `PhraseMatcher`

Además de los modelos estadísticos, spaCy incluye un motor de **reglas** para encontrar patrones
combinando atributos de tokens (texto, lema, POS, forma, etc.), algo así como "expresiones
regulares para objetos `Token`" en vez de para caracteres.

- **`Matcher`**: patrones basados en atributos de tokens (POS, lema, texto en minúsculas, ...).
- **`PhraseMatcher`**: búsqueda eficiente de listas largas de frases literales (nombres de
  empresas, productos, ciudades...), como si fuera un `Ctrl+F` inteligente.

In [21]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

# Patrón 1: cualquier ADJETIVO seguido de un SUSTANTIVO (p.ej. "rápido zorro")
matcher.add("ADJ_NOUN", [[{"POS": "ADJ"}, {"POS": "NOUN"}]])

# Patrón 2: el saludo literal "buenos días" (sin importar mayúsculas/minúsculas)
matcher.add("SALUDO", [[{"LOWER": "buenos"}, {"LOWER": "días"}]])

doc_ejemplo = nlp("El rápido zorro marrón salta sobre el perro perezoso. ¡Buenos días a todos!")

for match_id, inicio, fin in matcher(doc_ejemplo):
    regla = nlp.vocab.strings[match_id]
    print(f"{regla:10} -> {doc_ejemplo[inicio:fin].text!r}")

ADJ_NOUN   -> 'rápido zorro'
ADJ_NOUN   -> 'marrón salta'
SALUDO     -> 'Buenos días'


In [22]:
from spacy.matcher import PhraseMatcher

phrase_matcher = PhraseMatcher(nlp.vocab)
empresas = ["Apple", "Google", "Microsoft"]
patrones = [nlp.make_doc(nombre) for nombre in empresas]
phrase_matcher.add("EMPRESAS", patrones)

doc_empresas = nlp("Apple y Microsoft compiten con Google en inteligencia artificial.")
for match_id, inicio, fin in phrase_matcher(doc_empresas):
    print(nlp.vocab.strings[match_id], "->", doc_empresas[inicio:fin].text)

EMPRESAS -> Apple
EMPRESAS -> Microsoft
EMPRESAS -> Google


## 14. Procesamiento eficiente de muchos textos: `nlp.pipe`

Si necesitas procesar **muchos textos** (miles de filas de un CSV, por ejemplo), llamar a
`nlp(texto)` uno por uno en un ciclo funciona, pero es más lento que usar `nlp.pipe(lista_textos)`,
que procesa los textos por lotes internamente y es la forma recomendada para producción.

In [23]:
titulares = [
    "Madrid es la capital de España.",
    "El BBVA anunció resultados trimestrales.",
    "Lionel Messi juega en el Inter Miami.",
]

for doc_i in nlp.pipe(titulares):
    entidades = [(ent.text, ent.label_) for ent in doc_i.ents]
    print(f"{doc_i.text!r:45} -> {entidades}")

'Madrid es la capital de España.'             -> [('Madrid', 'LOC'), ('España', 'LOC')]
'El BBVA anunció resultados trimestrales.'    -> [('BBVA', 'ORG')]
'Lionel Messi juega en el Inter Miami.'       -> [('Lionel Messi', 'PER'), ('Inter Miami', 'ORG')]


## 15. Personalizando el *pipeline*: componentes y extensiones propias

spaCy permite:

1. **Agregar componentes propios** al pipeline con el decorador `@Language.component`.
2. **Agregar atributos personalizados** a `Doc`, `Span` o `Token` con `set_extension`, accesibles
   siempre bajo el namespace `._` (para no chocar con atributos nativos de spaCy).

Esto es la base para construir *pipelines* de NLP a la medida de un proyecto.

In [24]:
from spacy.language import Language
from spacy.tokens import Doc

# 1) Extensión personalizada: número de entidades encontradas en el documento
if not Doc.has_extension("num_entidades"):
    Doc.set_extension("num_entidades", getter=lambda doc: len(doc.ents))

# 2) Componente personalizado (aquí solo es un ejemplo simple; podría, p.ej.,
#    marcar el documento si contiene algún dinero mencionado)
@Language.component("marcar_organizaciones")
def marcar_organizaciones(doc):
    doc._.tiene_organizacion = any(ent.label_ == "ORG" for ent in doc.ents)
    return doc

if not Doc.has_extension("tiene_organizacion"):
    Doc.set_extension("tiene_organizacion", default=False)

if "marcar_organizaciones" not in nlp.pipe_names:
    nlp.add_pipe("marcar_organizaciones", last=True)

print("Pipeline actualizado:", nlp.pipe_names)

doc_final = nlp(texto)
print("Número de entidades:", doc_final._.num_entidades)
print("¿Menciona alguna organización?", doc_final._.tiene_organizacion)

Pipeline actualizado: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'marcar_organizaciones']
Número de entidades: 2
¿Menciona alguna organización? True


## 16. Serialización: guardar y cargar resultados (`DocBin`)

Volver a analizar el mismo texto cada vez que se necesita es costoso. spaCy permite **guardar**
objetos `Doc` ya procesados a disco (en binario, muy compacto) y **recuperarlos** después sin
tener que re-ejecutar el *pipeline*. La forma recomendada para colecciones de documentos es
`DocBin`.

In [25]:
from spacy.tokens import DocBin

doc_bin = DocBin()
doc_bin.add(doc)
doc_bin.add(doc_final)

# Guardar a disco:
# doc_bin.to_disk("documentos.spacy")

# Aquí lo hacemos en memoria para no dejar archivos sueltos en el tutorial:
datos = doc_bin.to_bytes()
print("Tamaño serializado:", len(datos), "bytes")

doc_bin_cargado = DocBin().from_bytes(datos)
docs_recuperados = list(doc_bin_cargado.get_docs(nlp.vocab))
print("Documentos recuperados:", len(docs_recuperados))
print("Primero:", docs_recuperados[0].text[:50], "...")

Tamaño serializado: 1059 bytes
Documentos recuperados: 2
Primero: Apple está buscando comprar una startup del Reino  ...


## 17. Proyecto integrador: extractor de información de una noticia

Cerramos combinando varias de las herramientas anteriores (`sents`, `ents`, `Matcher`) para
construir una función que recibe una noticia en texto plano y regresa un resumen estructurado:
número de oraciones, personas, organizaciones, lugares y si se detecta algún saludo/mención de
dinero mediante reglas propias.

In [26]:
from spacy.matcher import Matcher

nlp_extractor = spacy.load("es_core_news_sm")
matcher_dinero = Matcher(nlp_extractor.vocab)
# Patrón simple: un número seguido de una palabra de moneda/monto
matcher_dinero.add(
    "MONTO",
    [[{"LIKE_NUM": True}, {"LOWER": {"IN": ["millones", "millón", "dólares", "pesos", "euros"]}}]],
)


def extraer_informacion(texto_noticia):
    doc = nlp_extractor(texto_noticia)

    personas = [ent.text for ent in doc.ents if ent.label_ == "PER"]
    organizaciones = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    lugares = [ent.text for ent in doc.ents if ent.label_ == "LOC"]
    montos = [doc[i:f].text for _, i, f in matcher_dinero(doc)]

    return {
        "num_oraciones": len(list(doc.sents)),
        "num_tokens": len(doc),
        "personas": personas,
        "organizaciones": organizaciones,
        "lugares": lugares,
        "montos_detectados": montos,
    }


noticia = (
    "Apple está buscando comprar una startup del Reino Unido por mil millones de dólares. "
    "Según fuentes cercanas a Tim Cook, la operación se anunciará en Londres la próxima semana. "
    "Analistas del BBVA calculan que Microsoft y Google podrían responder con ofertas similares."
)

resultado = extraer_informacion(noticia)
for clave, valor in resultado.items():
    print(f"{clave:18}: {valor}")

num_oraciones     : 3
num_tokens        : 46
personas          : ['Según', 'Tim Cook']
organizaciones    : ['Apple', 'BBVA', 'Microsoft', 'Google']
lugares           : ['Reino Unido', 'Londres']
montos_detectados : ['mil millones']


> 🔎 **Nota sobre el resultado real:** el modelo etiquetó "Según" como `PER`, un error genuino
> de un modelo estadístico `sm`, no un bug del código. Es exactamente la clase de imprecisión que
> se menciona en la sección de limitaciones: conviene revisar y, si hace falta, corregir estos
> casos con reglas propias (`Matcher`/`EntityRuler`) antes de usar la salida en un sistema real.

## 18. Buenas prácticas, limitaciones y siguientes pasos

**Buenas prácticas**

- Usa `nlp.pipe(textos)` en lugar de un ciclo con `nlp(texto)` cuando proceses muchos documentos.
- Si no necesitas todo el pipeline (p.ej. solo tokenizar), desactiva componentes para ganar
  velocidad: `nlp = spacy.load("es_core_news_sm", disable=["ner", "parser"])`.
- Guarda resultados intermedios con `DocBin` en vez de re-procesar texto una y otra vez.
- Usa `spacy.explain(etiqueta)` cuando no recuerdes qué significa una abreviatura.

**Limitaciones a tener en cuenta**

- Los modelos son **estadísticos**: se equivocan (como vimos, algún adjetivo/sustantivo mal
  etiquetado es normal). Siempre valida en tu dominio específico.
- El esquema de entidades (`ent.label_`) cambia según el modelo/idioma cargado.
- Los modelos `sm` priorizan velocidad sobre precisión; para tareas críticas conviene evaluar
  `md`, `lg` o incluso modelos basados en *transformers* (`_trf`).

**Siguientes pasos**

- Curso oficial interactivo y gratuito: [course.spacy.io](https://course.spacy.io)
- Documentación: [spacy.io/usage](https://spacy.io/usage)
- `EntityRuler` / `SpanRuler`: reglas para mejorar o complementar el NER estadístico.
- `spacy train`: entrenar o afinar tus propios modelos con datos anotados.
- Integración con `transformers` (Hugging Face) mediante `spacy-transformers`.

## 19. Resumen / *cheatsheet* de referencia rápida

| Tarea | Código |
|---|---|
| Cargar un modelo | `nlp = spacy.load("es_core_news_sm")` |
| Procesar un texto | `doc = nlp(texto)` |
| Iterar tokens | `for token in doc: ...` |
| Lema / POS / dependencia | `token.lemma_`, `token.pos_`, `token.dep_` |
| ¿Es stop word? | `token.is_stop` |
| Oraciones | `doc.sents` |
| Entidades nombradas | `doc.ents`, `ent.text`, `ent.label_` |
| Sintagmas nominales | `doc.noun_chunks` |
| Similitud semántica | `token1.similarity(token2)` (requiere modelo `md`/`lg`) |
| Explicar una etiqueta | `spacy.explain("ORG")` |
| Patrones basados en reglas | `Matcher`, `PhraseMatcher` |
| Procesar muchos textos | `nlp.pipe(lista_de_textos)` |
| Componente/atributo propio | `@Language.component`, `Doc.set_extension` |
| Visualizar | `displacy.render(doc, style="dep"/"ent")` |
| Guardar/cargar documentos | `DocBin` |

---

**Fin del tutorial.** Con esto ya se tienen las bases suficientes para tokenizar, etiquetar,
analizar sintaxis, extraer entidades, comparar significados y construir *pipelines* propios con
spaCy, listo para empezar a desarrollar programas de NLP.